# 01 — Python Foundations, Types and Registries
## Writing AI software that stays easy to change


**Rule:** every instructional example is complete and executable. Only the final project is intentionally unfinished.

### How to use this notebook

Run the cells from top to bottom. Every code cell is finished and runnable. Only the last cell (the project) is left for you.

You will see:

- **Predict** — before you run a cell, write down what you expect it to print, and why.
- **What you just saw** — a short note after a cell that puts the output in plain words.
- **`assert` lines** — these are the specification. If one fails after you edit a cell, your change broke a rule the code is supposed to keep.

The project at the end is open: you get the goal and the acceptance criteria, and you write the code.

In [1]:
from __future__ import annotations

import random
from dataclasses import dataclass
from typing import Any, Callable, Iterable, Iterator, Protocol, Sequence

SEED = 7
random.seed(SEED)   # habit: seed randomness at the top of every program
print(f"Ready. Seed = {SEED}")

Ready. Seed = 7


### What the setup cell does

- `from __future__ import annotations` lets type hints refer to names that do not exist yet (like a method that returns its own class) without quoting them.
- The imports are the standard-library tools used below.
- `random.seed(SEED)` fixes the randomness. Nothing here depends on it, but it is a good habit — real experiments are not repeatable without it.

Run this cell first; every later cell needs its imports.

## Before we start: two ideas

This notebook builds two things that keep AI code understandable as it grows.

### A *type*

A **type** says what kind of value something is, and what you can do with it:

```python
name = "Ada"                    # str   -> text
batch_size = 32                 # int   -> whole number
confidence = 0.91               # float -> decimal number
tokens = [101, 202]             # list[int]  -> an ordered collection of ints
override = {"temperature": 0.2} # dict[str, float] -> named settings
```

A type hint like `list[float]` tells a reader (and tools like a type checker) what to expect. Python does **not** check it while the program runs. So real applications add their own checks at the edges — where data comes in from a file, an API, or a model.

### A *registry*

A **registry** maps a name to a piece of code. Instead of a growing chain of `if`:

```python
if model_name == "keyword":
    model = KeywordModel()
elif model_name == "sentiment":
    model = SentimentModel()
```

you register each option once and look it up by name:

```python
models = {"keyword": KeywordModel}
model = models["keyword"]()
```

Adding a model then means adding one line, not editing the chain. The sections below build the pieces that make this safe: input checking, mutability, protocols, dataclasses, and clear error messages.

## 1. Data has a shape, a type and a meaning

Python is *dynamically typed*: a value knows its own type while the program runs, and a name can point at different types over time. Hints describe intent; they do not enforce it. So for AI systems you check values yourself at the edges — files, APIs, model outputs.

Three separate questions about any incoming value:

- **Shape** — is it a dict, a list, a single value? Are the keys you need present?
- **Type** — can it be used as a `float`, a `str`, whatever you expect?
- **Meaning** — does it satisfy the rule of your problem? A probability must be between 0 and 1.

The function below throws away values it cannot use, instead of guessing. (An API often wants to *report* what was wrong instead — that is the next section.)

![Data contract pipeline](assets/data_contract.svg)

**Predict.** In the next cell, `raw_scores = ["0.91", None, "bad", "0.74", 1.2]`. Which of the five survive? Remember `float("nan")` does **not** raise — so what does `0.0 <= nan <= 1.0` return?

In [2]:
raw_scores = ["0.91", None, "bad", "0.74", 1.2]

def clean_probabilities(values: Iterable[object]) -> list[float]:
    cleaned = []
    for value in values:
        try:
            score = float(value)          # the edge: try to turn it into a number
        except (TypeError, ValueError):
            continue                       # None, "bad" -> skip
        if 0.0 <= score <= 1.0:
            cleaned.append(score)          # 1.2 is a number but out of range -> skip
    return cleaned

result = clean_probabilities(raw_scores)
print("input :", raw_scores)
print("kept  :", result)
print("dropped: None and 'bad' (not numbers), 1.2 (out of range)")

assert result == [0.91, 0.74]

input : ['0.91', None, 'bad', '0.74', 1.2]
kept  : [0.91, 0.74]
dropped: None and 'bad' (not numbers), 1.2 (out of range)


In [3]:
# The boundaries 0 and 1 are valid; everything malformed is skipped.
probability_cases = [0, 1, "0.25", "", "nan", -0.1, 1.1, object()]
cleaned_cases = clean_probabilities(probability_cases)

print("input :", probability_cases)
print("kept  :", cleaned_cases)
print('"nan" becomes float("nan"), and  0.0 <= nan <= 1.0  is False -> dropped')

assert cleaned_cases == [0.0, 1.0, 0.25]

input : [0, 1, '0.25', '', 'nan', -0.1, 1.1, <object object at 0x7ff9b38389f0>]
kept  : [0.0, 1.0, 0.25]
"nan" becomes float("nan"), and  0.0 <= nan <= 1.0  is False -> dropped


### What you just saw

`clean_probabilities` drops anything it cannot use and says nothing about it. That is fine for a best-effort filter.

The key `nan` detail: `float("nan")` succeeds, so `nan` gets past the `float()` step. But **every** comparison with `nan` is `False`, so `0.0 <= nan <= 1.0` is `False` and the range check drops it. You get this rejection for free.

### Dropping vs reporting

At an API boundary, the caller usually needs to know *what* was wrong and *where*, so they can fix their input. Same logic, but now it returns the accepted values **and** a list of structured rejections.

In [4]:
@dataclass(frozen=True)
class Rejection:
    index: int
    value: object
    reason: str

def validate_probabilities(values: Iterable[object]) -> tuple[list[float], list[Rejection]]:
    accepted: list[float] = []
    rejected: list[Rejection] = []
    for i, value in enumerate(values):
        try:
            score = float(value)
        except (TypeError, ValueError):
            rejected.append(Rejection(i, value, "not a number"))
            continue
        if not (0.0 <= score <= 1.0):      # also catches nan
            rejected.append(Rejection(i, value, "outside [0, 1]"))
            continue
        accepted.append(score)
    return accepted, rejected

accepted, rejected = validate_probabilities(["0.91", None, "bad", "0.74", 1.2, "nan"])
print("accepted:", accepted)
print("rejected:")
for r in rejected:
    print(f"  index {r.index}: {r.value!r:8} -> {r.reason}")

assert accepted == [0.91, 0.74]
assert [r.reason for r in rejected] == ["not a number", "not a number", "outside [0, 1]", "outside [0, 1]"]

accepted: [0.91, 0.74]
rejected:
  index 1: None     -> not a number
  index 2: 'bad'    -> not a number
  index 4: 1.2      -> outside [0, 1]
  index 5: 'nan'    -> outside [0, 1]


### What you just saw

Same accepted values as before, but now every dropped value comes back as a `Rejection` with its position and a reason. A caller can print that list and fix exactly the fields that failed. This is the pattern you want at the outer edge of a service.

## 2. Mutability and aliasing

Lists and dicts can be changed in place. Assigning one to a new name does **not** copy it — both names point at the same object.

- `x = y` — two names, one object. Change through either name, both see it.
- `y.copy()` — a *shallow* copy: a new outer dict, but the nested lists inside are still shared.
- `copy.deepcopy(y)` — copies all the way down.

**Predict.** After `alias["tokens"].append(30)`, what is in `shallow["tokens"]`? And in `independent["tokens"]`?

In [5]:
batch = {"tokens": [10, 20]}
alias = batch                                   # same object
shallow = batch.copy()                           # new dict, SAME inner list
independent = {"tokens": batch["tokens"].copy()} # new dict, new inner list

alias["tokens"].append(30)

print("batch      :", batch)
print("alias      :", alias, "  <- same object as batch")
print("shallow    :", shallow, "  <- shares the inner list, so it saw the append")
print("independent:", independent, "  <- has its own inner list, unchanged")

assert shallow["tokens"] == [10, 20, 30]
assert independent["tokens"] == [10, 20]

batch      : {'tokens': [10, 20, 30]}
alias      : {'tokens': [10, 20, 30]}   <- same object as batch
shallow    : {'tokens': [10, 20, 30]}   <- shares the inner list, so it saw the append
independent: {'tokens': [10, 20]}   <- has its own inner list, unchanged


In [6]:
import copy

deep = copy.deepcopy(batch)     # snapshot of batch RIGHT NOW: {"tokens": [10, 20, 30]}
batch["tokens"].append(40)      # change the original afterwards

print("deep    :", deep,    "  <- copied before the append(40), so still [10, 20, 30]")
print("shallow :", shallow, "  <- still sharing batch's list, so it saw the append(40)")

assert deep["tokens"] == [10, 20, 30]
assert shallow["tokens"] == [10, 20, 30, 40]

deep    : {'tokens': [10, 20, 30]}   <- copied before the append(40), so still [10, 20, 30]
shallow : {'tokens': [10, 20, 30, 40]}   <- still sharing batch's list, so it saw the append(40)


### The same bug, inside a function

Two shapes of this bug show up constantly:

1. a function that changes one of its arguments in place, and
2. a **mutable default argument** — a `[]` or `{}` in the function signature, which is created **once** and then shared by every call.

In [7]:
# 1. Changing an argument changes the caller's object too.
def add_bos(tokens: list[int], bos: int = 1) -> list[int]:
    tokens.insert(0, bos)          # in place!
    return tokens

caller_owned = [10, 20]
add_bos(caller_owned)
print("after add_bos, caller_owned =", caller_owned, " <- the caller's list was modified")
assert caller_owned == [1, 10, 20]

# A pure version builds a new list and leaves the input alone.
def with_bos(tokens: Sequence[int], bos: int = 1) -> list[int]:
    return [bos, *tokens]

safe_input = [10, 20]
print("with_bos returns   ", with_bos(safe_input))
print("safe_input is still", safe_input, " <- untouched")
assert safe_input == [10, 20]

after add_bos, caller_owned = [1, 10, 20]  <- the caller's list was modified
with_bos returns    [1, 10, 20]
safe_input is still [10, 20]  <- untouched


In [8]:
# 2. The mutable default argument trap.
def buggy_collect(item, sink=[]):        # sink=[] is created ONCE, at definition time
    sink.append(item)
    return sink

print("buggy_collect('a') ->", buggy_collect("a"))
print("buggy_collect('b') ->", buggy_collect("b"), " <- 'a' is still there!")
assert buggy_collect("c") == ["a", "b", "c"]

# The fix: default to None, make a fresh list inside.
def fixed_collect(item, sink=None):
    sink = [] if sink is None else sink
    sink.append(item)
    return sink

print("fixed_collect('a') ->", fixed_collect("a"))
print("fixed_collect('b') ->", fixed_collect("b"), " <- fresh list each call")
assert fixed_collect("b") == ["b"]

buggy_collect('a') -> ['a']
buggy_collect('b') -> ['a', 'b']  <- 'a' is still there!
fixed_collect('a') -> ['a']
fixed_collect('b') -> ['b']  <- fresh list each call


## 3. Comprehensions and generators

- A **list comprehension** `[f(x) for x in xs]` builds the whole list right away.
- A **generator** `(f(x) for x in xs)`, or a function with `yield`, produces one item at a time, only when asked. Good for large or endless streams.

A generator is **single-use**: once you have looped over it, it is empty. If you need the results more than once, make a list.

**Predict.** In the cell after next, `stream` is a fresh generator. What does the **third** assertion check — and why is it `== []`?

In [9]:
def token_windows(tokens: Sequence[int], width: int) -> Iterator[tuple[int, ...]]:
    if width <= 0:
        raise ValueError("width must be positive")
    for start in range(len(tokens) - width + 1):
        yield tuple(tokens[start:start + width])

windows = token_windows([11, 12, 13, 14], width=3)
print("type of windows:", type(windows).__name__, " <- a generator, nothing computed yet")
print("as a list      :", list(windows))
assert list(token_windows([11, 12, 13, 14], 3)) == [(11, 12, 13), (12, 13, 14)]

type of windows: generator  <- a generator, nothing computed yet
as a list      : [(11, 12, 13), (12, 13, 14)]


In [10]:
stream = token_windows([1, 2, 3, 4], width=2)

print("next(stream)      ->", next(stream))     # pulls one item
print("list(stream)      ->", list(stream))     # pulls the rest
print("list(stream) again->", list(stream), " <- empty: the generator is used up")

assert list(token_windows([1, 2, 3, 4], 2)) == [(1, 2), (2, 3), (3, 4)]

next(stream)      -> (1, 2)
list(stream)      -> [(2, 3), (3, 4)]
list(stream) again-> []  <- empty: the generator is used up


### What you just saw

Calling `token_windows(...)` runs none of the loop body — it hands back a generator. The body runs a step at a time as `next()` or a `for` loop pulls items. After the last item, the generator stays empty; you call the function again for a fresh one.

## 4. Dataclasses and protocols

- A **dataclass** writes the boring parts of a record class for you (`__init__`, `__repr__`, `==`).
- A **protocol** describes a class by the methods it has, not by what it inherits. Any class with a `predict(self, text) -> Prediction` method counts as a `Predictor`, no base class needed.

A protocol helps a type checker. It does **not** check anything while the program runs. The real contract is tested when `run_model` actually calls `predict` — and if that output crosses a trust boundary, you still validate it.

In [11]:
@dataclass(frozen=True)
class Prediction:
    label: str
    confidence: float

class Predictor(Protocol):
    def predict(self, text: str) -> Prediction: ...

class KeywordModel:
    def __init__(self, keyword: str = "urgent") -> None:
        self.keyword = keyword

    def predict(self, text: str) -> Prediction:
        hit = self.keyword.lower() in text.lower()
        return Prediction(self.keyword if hit else "normal", 0.9 if hit else 0.7)

def run_model(model: Predictor, texts: Iterable[str]) -> list[Prediction]:
    return [model.predict(t) for t in texts]

for p in run_model(KeywordModel(), ["Routine update", "URGENT: server down"]):
    print(p)

Prediction(label='normal', confidence=0.7)
Prediction(label='urgent', confidence=0.9)


### Why is `Prediction` frozen?

`frozen=True` means you cannot reassign a field after the object is made. Once a model has returned a `Prediction`, no later code can quietly change its `label` or `confidence`. (It does not deep-freeze — a mutable object *inside* a field could still be changed.)

In [12]:
from dataclasses import FrozenInstanceError

prediction = Prediction("normal", 0.7)
try:
    prediction.label = "urgent"          # not allowed on a frozen dataclass
except FrozenInstanceError:
    print("OK: a frozen Prediction refuses  prediction.label = ...")
else:
    raise AssertionError("A frozen Prediction should reject reassignment")

OK: a frozen Prediction refuses  prediction.label = ...


### Checking a protocol at runtime

A plain `Protocol` is invisible at runtime. Add `@runtime_checkable` and `isinstance` will work — but it only checks that the **method names** exist, not their signatures or return types.

**Predict.** `Impostor` has a `predict` method that returns a `str`, not a `Prediction`. Does `isinstance(Impostor(), PredictorRC)` pass?

In [13]:
from typing import runtime_checkable

@runtime_checkable
class PredictorRC(Protocol):
    def predict(self, text: str) -> Prediction: ...

class Impostor:
    def predict(self, text: str) -> str:      # right name, WRONG return type
        return "guess"

print("KeywordModel is a PredictorRC:", isinstance(KeywordModel(), PredictorRC))
print("Impostor     is a PredictorRC:", isinstance(Impostor(), PredictorRC), " <- it only checks the name 'predict'")
print("object()     is a PredictorRC:", isinstance(object(), PredictorRC))

assert isinstance(Impostor(), PredictorRC)         # passes on method name alone
assert not isinstance(object(), PredictorRC)

KeywordModel is a PredictorRC: True
Impostor     is a PredictorRC: True  <- it only checks the name 'predict'
object()     is a PredictorRC: False


### What you just saw

`runtime_checkable` answers "does it have a method called `predict`?" — not "does `predict` return the right thing?". `Impostor` passes the `isinstance` check and would still break your pipeline. If the output matters, validate the actual return value.

## 5. The registry

A `Registry` maps a name to a class. It replaces the growing `if/elif` chain and lets new code register itself. Two rules make it safe:

- **Reject duplicate names.** Silently overwriting one makes results depend on import order.
- **Make the "unknown name" error list the valid names**, so the caller can fix it.

`available()` returns a sorted list of the registered names — handy in error messages and tests.

In [14]:
class Registry:
    def __init__(self) -> None:
        self._items: dict[str, type] = {}

    def register(self, name: str) -> Callable[[type], type]:
        def decorator(cls: type) -> type:
            if name in self._items:
                raise KeyError(f"Already registered: {name}")
            self._items[name] = cls
            return cls
        return decorator

    def create(self, name: str, **kwargs: Any) -> Any:
        try:
            cls = self._items[name]
        except KeyError as exc:
            raise KeyError(f"Unknown component {name!r}; choose {self.available()}") from exc
        return cls(**kwargs)

    def available(self) -> list[str]:
        return sorted(self._items)

models = Registry()

@models.register("keyword")                      # <- registers the class below under "keyword"
class ConfigurableKeywordModel(KeywordModel):
    def __init__(self, keyword: str = "urgent"):
        super().__init__(keyword)

model = models.create("keyword", keyword="alert")
print("available models:", models.available())
print('models.create("keyword", keyword="alert").predict("Alert received") ->',
      model.predict("Alert received"))
assert model.predict("Alert received").label == "alert"

available models: ['keyword']
models.create("keyword", keyword="alert").predict("Alert received") -> Prediction(label='alert', confidence=0.9)


In [15]:
# Unknown name: the error tells you what IS available.
try:
    models.create("missing")
except KeyError as error:
    print("unknown name ->", error)
    assert "keyword" in str(error)

# Duplicate name: refused, not silently overwritten.
try:
    @models.register("keyword")
    class DuplicateKeywordModel(KeywordModel):
        pass
except KeyError as error:
    print("duplicate name ->", error)
    assert "Already registered" in str(error)

unknown name -> "Unknown component 'missing'; choose ['keyword']"
duplicate name -> 'Already registered: keyword'


### How the `@models.register("keyword")` line works

Python builds the class first, then hands it to `models.register("keyword")`. That call stores the class under `"keyword"` and returns it unchanged. Later, `models.create("keyword")` looks the class up and calls it. The `@` line is just: *"take the class defined below, pass it through this function."*

## 6. Validating a structured input

In production you would use a schema library (pydantic, attrs). The standard-library version below shows the steps plainly: read each field, convert its type, check its range, reject unknown tools, and raise a **specific** error that says what is wrong.

In [16]:
@dataclass(frozen=True)
class AgentAction:
    tool: str
    arguments: dict[str, Any]
    confidence: float

    @classmethod
    def from_dict(cls, data: dict[str, Any], allowed_tools: set[str]) -> "AgentAction":
        tool = str(data.get("tool", ""))
        if tool not in allowed_tools:
            raise ValueError(f"Tool {tool!r} is not allowed; allowed: {sorted(allowed_tools)}")

        raw_confidence = data.get("confidence", 0.0)
        try:
            confidence = float(raw_confidence)
        except (TypeError, ValueError):
            raise ValueError(f"confidence must be numeric, got {raw_confidence!r}") from None
        if not 0.0 <= confidence <= 1.0:          # also rejects nan
            raise ValueError("confidence must be in [0, 1]")

        arguments = data.get("arguments", {})
        if not isinstance(arguments, dict):
            raise TypeError("arguments must be a dictionary")

        return cls(tool, arguments, confidence)

action = AgentAction.from_dict(
    {"tool": "search", "arguments": {"q": "DAG"}, "confidence": "0.84"}, {"search"}
)
print("parsed:", action)
print('note: confidence "0.84" (a string) was converted to the float 0.84')

parsed: AgentAction(tool='search', arguments={'q': 'DAG'}, confidence=0.84)
note: confidence "0.84" (a string) was converted to the float 0.84


In [17]:
invalid_actions = [
    {"tool": "delete", "confidence": 0.8},                  # tool not allowed
    {"tool": "search", "confidence": 1.5},                  # out of range
    {"tool": "search", "arguments": [], "confidence": 0.8}, # arguments wrong type
    {"tool": "search", "confidence": "high"},               # not a number
    {"tool": "search", "confidence": None},                 # null
    {"tool": "search", "confidence": float("nan")},         # nan
]

for bad in invalid_actions:
    try:
        AgentAction.from_dict(bad, {"search"})
    except (TypeError, ValueError) as exc:
        print(f"{str(bad):55} -> {type(exc).__name__}: {exc}")
    else:
        raise AssertionError(f"Expected a failure for {bad}")

{'tool': 'delete', 'confidence': 0.8}                   -> ValueError: Tool 'delete' is not allowed; allowed: ['search']
{'tool': 'search', 'confidence': 1.5}                   -> ValueError: confidence must be in [0, 1]
{'tool': 'search', 'arguments': [], 'confidence': 0.8}  -> TypeError: arguments must be a dictionary
{'tool': 'search', 'confidence': 'high'}                -> ValueError: confidence must be numeric, got 'high'
{'tool': 'search', 'confidence': None}                  -> ValueError: confidence must be numeric, got None
{'tool': 'search', 'confidence': nan}                   -> ValueError: confidence must be in [0, 1]


### What you just saw

Every malformed input raised a clear, specific error naming the problem — not a generic crash three functions later. That is the point of validating at the boundary: the failure happens where the bad data enters, with a message the caller can act on.

## Project — Extensible inference gateway

Build a registry-backed gateway with two predictors, validated configurations, structured errors and tests.

**Suggested test matrix:**

- Registering a unique name succeeds; registering it twice raises a clear error.
- Creating an unknown model raises an error listing available names.
- Both predictors satisfy the `Predictor` protocol and return `Prediction` objects.
- Confidence values `0` and `1` are accepted; negative, greater-than-one, non-numeric and `nan` values are rejected.
- The gateway does not mutate the input text collection or configuration dictionaries.
- Adding a third model requires only a registration and its own tests, not a gateway change.

**Acceptance criteria:** unknown and duplicate names fail clearly; invalid confidence values are rejected; adding a third model does not modify the gateway; inputs are not mutated.

You may `from course_utils import Registry, Prediction, Predictor, AgentAction` instead of copying the cells above; the module ships the same implementations with docstrings.

**Checks to run yourself**

- Pass a config dict in, then assert the dict is unchanged after the call.
- Register a third predictor in a separate cell and confirm no gateway code changed.
- Drive `confidence` through `0`, `1`, `-0.001`, `1.5`, `"0.5"`, `None`, `float("nan")` and check each verdict.
- Make one predictor raise inside `predict`; decide whether the gateway propagates or wraps the error, and test that decision.

In [ ]:
# PROJECT WORKSPACE — intentionally incomplete
#
# Reuse Registry, Prediction, the Predictor protocol and AgentAction-style validation
# from this notebook, or import them:
#     from course_utils import Registry, Prediction, Predictor, AgentAction
#
# 1. Define a second predictor (e.g. a length-threshold or word-list model) that
#    returns Prediction and satisfies Predictor.
# 2. Implement InferenceGateway:
#      - build it from a config mapping {name: kwargs}
#      - resolve models through a Registry by name
#      - validate confidence; reject unknown / duplicate names with clear errors
#      - never mutate the caller's config or text inputs
# 3. Encode the test matrix from the brief as assert-based tests.

class InferenceGateway:
    ...


raise NotImplementedError("Implement and test the extensible inference gateway")
